In [ ]:
import subprocess
import os
from os.path import join
import pandas as pd
from datetime import datetime

import intake, intake_esm

from IPython.display import display, HTML

from urllib.parse import quote


import numpy as np
import json

In [ ]:
# Gather all information in txt file
data_path = '/scratchu/yrobin/BC2602/ADJUST/'
subprocess.call(f"ls -R {data_path} | perl -pe 's/(.*)_(.*)_(.*)_(.*)_(.*)_(.*)_(.*)_(.*).nc/\u$1,$2,$3,$4,$5,$6,$7,$8,$&/' > CORDEX_Adjust_catalog.txt", shell=True)

In [2]:
work_dir = "/home/user/These/cordex_htws_cc3d"
with open(join(work_dir,'Data','CORDEX_Adjust_catalog.txt'),'r') as file:
    lines = file.readlines()

In [3]:
lines

['/scratchu/yrobin/BC2602/ADJUST/:\n',
 'CNRM-CERFACS-CNRM-CM5\n',
 'ICHEC-EC-EARTH\n',
 'MOHC-HadGEM2-ES\n',
 'MPI-M-MPI-ESM-LR\n',
 'NCC-NorESM1-M\n',
 '\n',
 '/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CNRM-CM5:\n',
 'CNRM-ALADIN63\n',
 '\n',
 '/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CNRM-CM5/CNRM-ALADIN63:\n',
 'historical\n',
 'rcp45\n',
 'rcp85\n',
 '\n',
 '/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CNRM-CM5/CNRM-ALADIN63/historical:\n',
 'r1i1p1\n',
 '\n',
 '/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CNRM-CM5/CNRM-ALADIN63/historical/r1i1p1:\n',
 'day\n',
 '\n',
 '/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CNRM-CM5/CNRM-ALADIN63/historical/r1i1p1/day:\n',
 'hursAdjust\n',
 'prAdjust\n',
 'sfcWindAdjust\n',
 'tasAdjust\n',
 'tasmaxAdjust\n',
 'tasminAdjust\n',
 '\n',
 '/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CNRM-CM5/CNRM-ALADIN63/historical/r1i1p1/day/hursAdjust:\n',
 'HursAdjust,IPSL-CDFt,CNRM-CERFACS-CNRM-CM5,CNRM-ALADIN63,r1i1p1,historical,day,19510101-20051231,hu

In [58]:
columns = ['path','project','product','domain','driving_model','experiment','ensemble','rcm_model','rcm_version',
'time_frequency','variable','version','period_start','period_end','latest','correction','reference_dataset']
df = pd.DataFrame(data=None,columns=columns)

In [59]:
count=0
for i in range(len(lines)):
    if '.nc' in lines[i]:
        data = lines[i][:-2].split(",") # remove '\n'
        previous_line = lines[i-1][:-2] # Get previous line and remove ':\n'
        df.loc[count,'path'] = previous_line+'/'+data[-1]
        df.loc[count,'driving_model'] = data[2]
        df.loc[count,'experiment'] = data[5]
        df.loc[count,'ensemble'] = data[4]
        df.loc[count,'rcm_model'] = data[3]
        df.loc[count,'variable'] = previous_line.split('/')[-1][:-6]
        df.loc[count,'period_start'] = datetime.strptime(data[7].split("-")[0],'%Y%m%d')
        df.loc[count,'period_end'] = datetime.strptime(data[7].split("-")[1],'%Y%m%d')
        count+=1
df['project'] = 'CORDEX'
df['product'] = 'ADJUST'
df['domain'] = 'EUR-11'
df['time_frequency'] = 'day'
df['version'] = 'BC2602'
df['rcm_version'] = 'BC2602'
df['latest'] = True
df['correction'] = 'IPSL-CDFt'
df['reference_dataset'] = 'ERA5'

In [62]:
df.to_csv(join(work_dir,'CORDEX_ADJUST.csv'))

In [61]:
df

,path,project,product,domain,driving_model,experiment,ensemble,rcm_model,rcm_version,time_frequency,variable,version,period_start,period_end,latest,correction,reference_dataset
0,/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CN...,CORDEX,ADJUST,EUR-11,CNRM-CERFACS-CNRM-CM5,historical,r1i1p1,CNRM-ALADIN63,BC2602,day,hurs,BC2602,1951-01-01 00:00:00,2005-12-31 00:00:00,True,IPSL-CDFt,ERA5
1,/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CN...,CORDEX,ADJUST,EUR-11,CNRM-CERFACS-CNRM-CM5,historical,r1i1p1,CNRM-ALADIN63,BC2602,day,pr,BC2602,1951-01-01 00:00:00,2005-12-31 00:00:00,True,IPSL-CDFt,ERA5
2,/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CN...,CORDEX,ADJUST,EUR-11,CNRM-CERFACS-CNRM-CM5,historical,r1i1p1,CNRM-ALADIN63,BC2602,day,sfcWind,BC2602,1951-01-01 00:00:00,2005-12-31 00:00:00,True,IPSL-CDFt,ERA5
3,/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CN...,CORDEX,ADJUST,EUR-11,CNRM-CERFACS-CNRM-CM5,historical,r1i1p1,CNRM-ALADIN63,BC2602,day,tas,BC2602,1951-01-01 00:00:00,2005-12-31 00:00:00,True,IPSL-CDFt,ERA5
4,/scratchu/yrobin/BC2602/ADJUST/CNRM-CERFACS-CN...,CORDEX,ADJUST,EUR-11,CNRM-CERFACS-CNRM-CM5,historical,r1i1p1,CNRM-ALADIN63,BC2602,day,tasmax,BC2602,1951-01-01 00:00:00,2005-12-31 00:00:00,True,IPSL-CDFt,ERA5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,/scratchu/yrobin/BC2602/ADJUST/NCC-NorESM1-M/G...,CORDEX,ADJUST,EUR-11,NCC-NorESM1-M,rcp85,r1i1p1,GERICS-REMO2015,BC2602,day,pr,BC2602,2006-01-01 00:00:00,2100-12-31 00:00:00,True,IPSL-CDFt,ERA5
160,/scratchu/yrobin/BC2602/ADJUST/NCC-NorESM1-M/G...,CORDEX,ADJUST,EUR-11,NCC-NorESM1-M,rcp85,r1i1p1,GERICS-REMO2015,BC2602,day,sfcWind,BC2602,2006-01-01 00:00:00,2100-12-31 00:00:00,True,IPSL-CDFt,ERA5
161,/scratchu/yrobin/BC2602/ADJUST/NCC-NorESM1-M/G...,CORDEX,ADJUST,EUR-11,NCC-NorESM1-M,rcp85,r1i1p1,GERICS-REMO2015,BC2602,day,tas,BC2602,2006-01-01 00:00:00,2100-12-31 00:00:00,True,IPSL-CDFt,ERA5
162,/scratchu/yrobin/BC2602/ADJUST/NCC-NorESM1-M/G...,CORDEX,ADJUST,EUR-11,NCC-NorESM1-M,rcp85,r1i1p1,GERICS-REMO2015,BC2602,day,tasmax,BC2602,2006-01-01 00:00:00,2100-12-31 00:00:00,True,IPSL-CDFt,ERA5
